In [71]:
import os, pickle, json
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
import spacy
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import numpy as np

In [72]:
# Leer chunks, está en excel
chunks_etiquetados = pd.read_excel("../data/results/chunks_etiquetados_binario.xlsx")
chunks_etiquetados.head()

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.411230,0.368818,0.363447,0.354240,0.378154,0.343397,0.387251,...,0.392998,0.423331,0.337017,0.352550,0.328395,0.394268,0.393815,0.389038,"[('ninguna', 0)]",0
1,1,1,"al mismo tiempo lo exponen, en ciertas ocasion...",0.331604,0.304309,0.346682,0.315253,0.344347,0.323262,0.344607,...,0.318409,0.378231,0.317467,0.368125,0.306375,0.381964,0.337864,0.306510,"[('ninguna', 0)]",0
2,2,1,los acuerdos con las Farc. Anunció que no prom...,0.356805,0.311163,0.303825,0.287517,0.329738,0.290927,0.332603,...,0.292858,0.405040,0.291387,0.322885,0.297822,0.338846,0.294842,0.318955,"[('ninguna', 0)]",0
3,3,1,moratoria en la explotación tipo fracking. Y f...,0.435032,0.397881,0.372960,0.368983,0.415538,0.357000,0.410173,...,0.392092,0.445708,0.370653,0.367256,0.341832,0.404120,0.375238,0.397525,"[('ninguna', 0)]",0
4,0,2,Las interpretaciones de la historia sirven com...,0.352643,0.367811,0.375234,0.357391,0.368184,0.347770,0.392265,...,0.368177,0.428733,0.353389,0.405911,0.371109,0.412153,0.397781,0.339795,"[('ninguna', 0)]",0


In [73]:
chunks_ciencias = chunks_etiquetados[chunks_etiquetados["etiqueta_ciencia"] == 1]
chunks_ciencias.shape

(7267, 22)

In [74]:
# columnas que contienen los scores de las diferentes ciencias
cols = [c for c in chunks_ciencias.columns
    if c.startswith("Ciencias_") or c.startswith("Ciencia_")]

# obtener sólo los chunks cuyo máximo absoluto cae en
# “Ciencia_Administracion_ciencia_investigacion”
chunks_ciencias_admin = chunks_ciencias[
    chunks_ciencias[cols].idxmax(axis=1)
    == "Ciencia_Administracion_ciencia_investigacion"
]

chunks_ciencias_admin.shape, chunks_ciencias_admin.head()

((3195, 22),
      chunk_id  id_doc                                        texto_chunk  \
 43          1      10  estabilidad, visión de futuro y apertura al co...   
 44          2      10  que se escoja la fórmula presidencial con estr...   
 46          0      11  El método es sencillo. Primero tensar un poco ...   
 72          1      15  creando poderosos cuerpos armados paralelos qu...   
 124         0      27  ¿Qué hace que miembros de sociedades divididas...   
 
      Ciencias_ambientales_ingenieria  Ciencias_espacio  Ciencias_fisicas  \
 43                          0.461249          0.396793          0.401036   
 44                          0.432813          0.389536          0.411760   
 46                          0.422972          0.398423          0.410561   
 72                          0.417950          0.416532          0.439040   
 124                         0.378887          0.372949          0.395535   
 
      Ciencias_Geografia_oceanografia  Ciencias_medicas  Ci

In [75]:
chunks_ciencias_admin

,chunk_id,id_doc,texto_chunk,Ciencias_ambientales_ingenieria,Ciencias_espacio,Ciencias_fisicas,Ciencias_Geografia_oceanografia,Ciencias_medicas,Ciencias_metereologia,Ciencias_naturales,...,Ciencias_tierra,Ciencia_Administracion_ciencia_investigacion,Ciencia_biologia,Ciencia_enfoque_cientifico,Ciencia_hidrologia,Ciencia_matematicas_estadistica,Ciencia_patologia,Ciencia_recursos_naturales,categorias_detectadas,etiqueta_ciencia
43,1,10,"estabilidad, visión de futuro y apertura al co...",0.461249,0.396793,0.401036,0.404579,0.445807,0.404155,0.421193,...,0.404820,0.500761,0.399866,0.407591,0.398157,0.456001,0.427080,0.436823,"[('Ciencias_polucion_catastrofes_seguridad', 0...",1
44,2,10,que se escoja la fórmula presidencial con estr...,0.432813,0.389536,0.411760,0.408603,0.419465,0.404222,0.437402,...,0.402378,0.486791,0.415309,0.440626,0.405430,0.466137,0.408754,0.426791,[('Ciencia_Administracion_ciencia_investigacio...,1
46,0,11,El método es sencillo. Primero tensar un poco ...,0.422972,0.398423,0.410561,0.395789,0.412290,0.392898,0.435677,...,0.381483,0.475857,0.407663,0.459802,0.403404,0.444334,0.411773,0.415889,[('Ciencia_Administracion_ciencia_investigacio...,1
72,1,15,creando poderosos cuerpos armados paralelos qu...,0.417950,0.416532,0.439040,0.376707,0.450884,0.383261,0.438462,...,0.393280,0.480262,0.422196,0.407696,0.385595,0.418124,0.449374,0.385799,[('Ciencia_Administracion_ciencia_investigacio...,1
124,0,27,¿Qué hace que miembros de sociedades divididas...,0.378887,0.372949,0.395535,0.357060,0.384036,0.360637,0.391136,...,0.372665,0.472678,0.370129,0.439334,0.360454,0.421877,0.367899,0.327612,[('Ciencia_Administracion_ciencia_investigacio...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62606,2,13670,de mercado para proyectos semejantes donde los...,0.405087,0.363564,0.348389,0.319041,0.377561,0.359642,0.365868,...,0.363907,0.493275,0.365584,0.427741,0.387962,0.391810,0.359039,0.356897,[('Ciencia_Administracion_ciencia_investigacio...,1
62607,3,13670,"ejecución. Por si fuera poco, durante el escas...",0.434345,0.389919,0.386449,0.331937,0.385795,0.381422,0.389492,...,0.342314,0.472417,0.381446,0.446409,0.379387,0.397558,0.401377,0.370059,[('Ciencia_Administracion_ciencia_investigacio...,1
62615,6,13671,empresas y juntos diseñaron el mejor marco pos...,0.409905,0.352519,0.373904,0.322059,0.379840,0.333315,0.378122,...,0.341531,0.484518,0.361957,0.385061,0.351471,0.377240,0.351933,0.359391,[('Ciencia_Administracion_ciencia_investigacio...,1
62620,4,13672,negociaciones. Tantas cosas por resolver y en ...,0.431984,0.397463,0.385541,0.378124,0.411482,0.375245,0.432138,...,0.375919,0.489522,0.368565,0.418586,0.397724,0.412351,0.419973,0.379041,[('Ciencia_Administracion_ciencia_investigacio...,1


In [76]:
# Leer embeddings gemma
embeddings = np.load(r"../data/processed/chunk_embeddings.npy")
embeddings.shape

(62651, 768)

In [77]:
# crear un boolean mask sobre el array completo de embeddings
# que marque las filas que están en `chunks_ciencias_admin`
mask_admin = chunks_etiquetados.index.isin(chunks_ciencias_admin.index)

# aplicar la máscara a los embeddings
embeddings_ciencias_admin = embeddings[mask_admin]


# Stopwords

In [78]:
# Cargar modelo pequeño de spaCy (rápido)
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])

# Stopwords en español desde spaCy
spanish_stopwords = nlp.Defaults.stop_words

# BERTopic

In [79]:
documents = chunks_ciencias_admin['texto_chunk'].tolist()
len(documents)

3195

In [80]:
vectorizer_model = CountVectorizer(
    stop_words= list(spanish_stopwords),
    max_features=5000,   
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 1),
    strip_accents=None
)

In [81]:
def get_topic_words(topic_model, top_n=10):
    topics_words = []

    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue  # excluimos outliers

        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)

    return topics_words

In [82]:
# Función que junta ambos métodos

def compute_bertopic_coherence(
    topic_model,
    documents,
    vectorizer_model,
    top_n_words=10
):
    analyzer = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]

    dictionary = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)

    corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

    topics_words = get_topic_words(topic_model, top_n=top_n_words)

    coherence_model = CoherenceModel(
        topics=topics_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        corpus=corpus,
        coherence='c_v'
    )

    return coherence_model.get_coherence(), coherence_model.get_coherence_per_topic()

In [83]:
nr_topics_values = [10, 20, 30, 40,  "auto"]
results = []

for nr in nr_topics_values:
    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=10,
        calculate_probabilities=False,
        verbose=True,
        language="spanish"
    )

    topics, probs = topic_model.fit_transform(
        documents,
        embeddings=embeddings_ciencias_admin   # embeddings Gemma
    )

     # ← Solo reducir outliers si existen
    if -1 in topics:
        new_topics = topic_model.reduce_outliers(
            documents,
            topics,
            strategy="c-tf-idf",
            embeddings=embeddings_ciencias_admin
        )
        topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)
    else:
        print(f"nr_topics={nr}: sin outliers, no se reduce.")

    coherence,_ = compute_bertopic_coherence(topic_model, documents, vectorizer_model)

    n_outliers = sum(1 for t in topic_model.topics_ if t == -1)


    results.append({
        "nr_topics": nr,
        "n_topics_final": len([t for t in topic_model.get_topics() if t != -1]),
        "coherence_c_v": coherence,
        "%_outliers": n_outliers / len(documents)
    })


2026-03-05 09:31:22,612 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-05 09:31:37,372 - BERTopic - Dimensionality - Completed ✓
2026-03-05 09:31:37,372 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-05 09:31:37,500 - BERTopic - Cluster - Completed ✓
2026-03-05 09:31:37,502 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-05 09:31:37,958 - BERTopic - Representation - Completed ✓
2026-03-05 09:31:37,958 - BERTopic - Topic reduction - Reducing number of topics
2026-03-05 09:31:37,981 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-05 09:31:38,443 - BERTopic - Representation - Completed ✓
2026-03-05 09:31:38,446 - BERTopic - Topic reduction - Reduced number of topics from 46 to 10
2026-03-05 09:31:38,794 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure tha

In [84]:
df_results = pd.DataFrame(results)
df_results.sort_values("coherence_c_v", ascending=False)

,nr_topics,n_topics_final,coherence_c_v,%_outliers
4,auto,32,0.652492,0.0
2,30,29,0.642005,0.0
1,20,19,0.640077,0.0
3,40,39,0.639298,0.0
0,10,9,0.558448,0.0


In [95]:
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=32,
    min_topic_size=10,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)
topics, probs = topic_model.fit_transform(
    documents,
    embeddings=embeddings_ciencias_admin   # embeddings Gemma
)

 # ← Solo reducir outliers si existen
if -1 in topics:
    new_topics = topic_model.reduce_outliers(
        documents,
        topics,
        strategy="c-tf-idf",
        embeddings=embeddings_ciencias_admin
    )
    topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)
else:
    print(f"nr_topics={nr}: sin outliers, no se reduce.")


2026-03-05 09:46:33,806 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-05 09:46:51,792 - BERTopic - Dimensionality - Completed ✓
2026-03-05 09:46:51,792 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-05 09:46:51,933 - BERTopic - Cluster - Completed ✓
2026-03-05 09:46:51,933 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-05 09:46:52,509 - BERTopic - Representation - Completed ✓
2026-03-05 09:46:52,509 - BERTopic - Topic reduction - Reducing number of topics
2026-03-05 09:46:52,548 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-05 09:46:53,015 - BERTopic - Representation - Completed ✓
2026-03-05 09:46:53,015 - BERTopic - Topic reduction - Reduced number of topics from 58 to 32
2026-03-05 09:46:53,675 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure tha

In [98]:
tm_res = topic_model.get_topic_info()
tm_res

Topic  Count                                            Name  \
0       0    515     0_economía_crecimiento_productividad_sector   
1       1    521      1_político_democracia_políticos_presidente   
2       2    191          2_datos_información_digital_tecnología   
3       3    173                   3_pandemia_salud_crisis_virus   
4       4    184  4_universidad_nacional_formación_investigación   
..    ...    ...                                             ...   
26     26     21                     26_china_unidos_india_mundo   
27     27     18               27_europea_unión_migrantes_europa   
28     28     19         28_oficina_asesores_presidente_perfiles   
29     29     17   29_independencia_bicentenario_expedición_paso   
30     30     13            30_redes_tendencias_falsas_contenido   

                                                                                                                Representation  \
0                          [economía, crecimiento, productividad, sector, fiscal, empleo, producción, rural, banco, inversión]   
1                   [político, democracia, políticos, presidente, duque, sociedad, centro, gobernabilidad, sociales, partidos]   
2                      [datos, información, digital, tecnología, internet, personas, tic, vigilancia, seguridad, inteligencia]   
3                                         [pandemia, salud, crisis, virus, economía, medidas, cuarentena, mundo, tiempo, vida]   
4     [universidad, nacional, formación, investigación, comunidades, educación, comunidad, pedagógica, calidad, universidades]   
..                                                                                                                         ...   
26                                    [china, unidos, india, mundo, aranceles, occidente, comercio, rusia, económica, asuntos]   
27                         [europea, unión, migrantes, europa, reclaman, países, francés, continente, gobiernos, comunitarias]   
28  [oficina, asesores, presidente, perfiles, formación, internacional, problemática, experiencia, equipo, interdisciplinaria]   
29               [independencia, bicentenario, expedición, paso, botánica, historia, eslabón, columna, observatorio, proyecto]   
30                              [redes, tendencias, falsas, contenido, sociales, medios, digital, interacción, real, noticias]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [99]:
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
topic_ids = [
    topic_id
    for topic_id in topic_model.get_topics().keys()
    if topic_id != -1
]
coherence_df = pd.DataFrame({
    "Topic": topic_ids,
    "Coherence_c_v": coh_per_topic
})

coherence_table = (
    tm_res[tm_res["Topic"] != -1]
    .merge(coherence_df, on="Topic", how="left")
)

In [100]:
total_docs = tm_res["Count"].sum()

In [101]:
top = (
    tm_res
    .sort_values("Count", ascending=False)
    .head(20)
    .copy()
)
top["Keywords"] = top["Representation"].apply(
    lambda words: ", ".join(words)
)


In [102]:
top["Percentage"] = (top["Count"] / total_docs * 100).round(2)

top = top.merge(
    coherence_df,
    on="Topic",
    how="left"
)


In [103]:
final_table = top[[
    "Topic",
    #"Count",
    "Coherence_c_v",
    "Percentage",
    "Keywords"
]]

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,1,0.566847,16.31,"político, democracia, políticos, presidente, duque, sociedad, centro, gobernabilidad, sociales, partidos"
1,0,0.675547,16.12,"economía, crecimiento, productividad, sector, fiscal, empleo, producción, rural, banco, inversión"
2,2,0.690051,5.98,"datos, información, digital, tecnología, internet, personas, tic, vigilancia, seguridad, inteligencia"
3,4,0.762485,5.76,"universidad, nacional, formación, investigación, comunidades, educación, comunidad, pedagógica, calidad, universidades"
4,3,0.550910,5.41,"pandemia, salud, crisis, virus, economía, medidas, cuarentena, mundo, tiempo, vida"
5,8,0.716247,4.69,"universidades, educación, superior, calidad, estudiantes, públicas, universidad, pública, formación, sistema"
6,12,0.614592,4.13,"paz, policía, seguridad, violencia, implementación, territorios, fuerzas, armas, acuerdos, farc"
7,6,0.593736,4.07,"corrupción, control, funcionarios, públicos, entidades, congreso, público, pública, congresistas, presupuesto"
8,7,0.510011,3.82,"justicia, electoral, judicial, magistrados, reforma, registraduría, derecho, voto, consejo, proceso"
9,11,0.697445,3.54,"ciencia, innovación, ministerio, tecnología, investigación, regalías, colciencias, sistema, ley, nacional"


In [104]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", None)

final_table

,Topic,Coherence_c_v,Percentage,Keywords
0,1,0.566847,16.31,"político, democracia, políticos, presidente, duque, sociedad, centro, gobernabilidad, sociales, partidos"
1,0,0.675547,16.12,"economía, crecimiento, productividad, sector, fiscal, empleo, producción, rural, banco, inversión"
2,2,0.690051,5.98,"datos, información, digital, tecnología, internet, personas, tic, vigilancia, seguridad, inteligencia"
3,4,0.762485,5.76,"universidad, nacional, formación, investigación, comunidades, educación, comunidad, pedagógica, calidad, universidades"
4,3,0.550910,5.41,"pandemia, salud, crisis, virus, economía, medidas, cuarentena, mundo, tiempo, vida"
5,8,0.716247,4.69,"universidades, educación, superior, calidad, estudiantes, públicas, universidad, pública, formación, sistema"
6,12,0.614592,4.13,"paz, policía, seguridad, violencia, implementación, territorios, fuerzas, armas, acuerdos, farc"
7,6,0.593736,4.07,"corrupción, control, funcionarios, públicos, entidades, congreso, público, pública, congresistas, presupuesto"
8,7,0.510011,3.82,"justicia, electoral, judicial, magistrados, reforma, registraduría, derecho, voto, consejo, proceso"
9,11,0.697445,3.54,"ciencia, innovación, ministerio, tecnología, investigación, regalías, colciencias, sistema, ley, nacional"
